# Paso 1 del tribunal: guardar todos los candidatos

Idea de jhonatan (20 oct 2026): el detector ya no decide, solo propone. Dispara "a discreción" y un tribunal
de jueces decide (ver `docs/arquitectura-tribunal.md`). Este notebook **no entrena nada**:

1. Pasa el detector de la ronda 3 en **modo generoso** (confianza desde 0,1 en vez de 0,8) por las 121 imágenes
   del examen y por las 586 de estudio.
2. De cada candidato guarda su silueta (del detector y del refinador), su ficha (confianza, tamaño, oscuridad,
   contraste con su entorno, forma, distancia al borde) y, en el examen, un recorte para el juez A.
3. Imprime la **tabla de acierto por franja de confianza** y cuántos filamentos reales están al alcance.
4. Comprueba que la cadena de la ronda 3 sigue dando ~0,4405 (si no, algo está mal cargado).

No genera `submission.csv`. Requisitos en Kaggle: *Accelerator* = GPU T4, y como *Input*: la competición y el
*Output* del notebook `filamentos-detector-ronda-2` (con `detector_mejor.pt` y `refinador.pt`).

In [ ]:
# Configuración. PRUEBA_RAPIDA=True reduce todo para comprobar que el notebook funciona.
PRUEBA_RAPIDA = False

RES = 1024              # resolución del detector general (la de la ronda 3)
CONF_MIN = 0.1          # el detector propone todo lo que tenga al menos esta confianza
UMBRAL_REF = 0.5        # umbral del refinador de la ronda 3
LADO_RECORTE = 128      # recortes para el juez A (solo examen)
CON_ESTUDIO = True      # también las 586 imágenes de estudio (sin recortes)

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=True)
    import segmentation_models_pytorch as smp
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

## Código del proyecto
Copiado del repositorio `filamentos-solares-2026`.

In [ ]:
import os
os.makedirs('filamentos', exist_ok=True)

In [ ]:
%%writefile filamentos/__init__.py
"""Código del proyecto de segmentación de filamentos solares."""

In [ ]:
%%writefile filamentos/datos.py
"""Lectura de MAGFiLO y conversión entre polígonos, máscaras y RLE."""
import json
import os
from collections import defaultdict

import numpy as np
from pycocotools import mask as mask_utils

TAM = 2048


def cargar_anotaciones(raiz):
    """Devuelve {nombre_imagen: {anotador: [anotación COCO, ...]}}.

    Una imagen puede tener hasta 3 registros, uno por anotador. El id del registro
    es "GGGGAA-<nombre>"; usamos GGGGAA como identificador del anotador.
    """
    with open(os.path.join(raiz, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")) as f:
        coco = json.load(f)
    registro = {i["id"]: (i["file_name"], i["id"].split("-")[0]) for i in coco["images"]}
    out = defaultdict(lambda: defaultdict(list))
    for reg_id, (nombre, anot) in registro.items():
        out[nombre][anot]  # imágenes anotadas sin filamentos también cuentan
    for a in coco["annotations"]:
        nombre, anot = registro[a["image_id"]]
        out[nombre][anot].append(a)
    return {k: dict(v) for k, v in out.items()}


def poligono_a_mascara(segmentacion, alto=TAM, ancho=TAM):
    rles = mask_utils.frPyObjects(segmentacion, alto, ancho)
    return mask_utils.decode(mask_utils.merge(rles)).astype(bool)


def mascaras_de(anotaciones):
    return [poligono_a_mascara(a["segmentation"]) for a in anotaciones]


def mascara_a_rle(mascara):
    """Cadena de 'counts' RLE de COCO, el formato de la columna segmentation_rle."""
    rle = mask_utils.encode(np.asfortranarray(mascara.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def mascara_a_poligonos(mascara, tolerancia=1.0):
    """Contornos de una máscara como listas [x0, y0, x1, y1, ...] (formato COCO), para el visor."""
    from scipy import ndimage as ndi
    from skimage import measure

    ys, xs = ndi.find_objects(mascara.astype(np.uint8))[0]
    local = np.pad(mascara[ys, xs], 1)
    poligonos = []
    for c in measure.find_contours(local.astype(np.uint8), 0.5):
        c = measure.approximate_polygon(c, tolerancia)
        if len(c) >= 3:
            poligonos.append([round(float(v), 1) for y, x in c for v in (x + xs.start - 1, y + ys.start - 1)])
    return poligonos


def rle_a_mascara(counts, alto=TAM, ancho=TAM):
    return mask_utils.decode({"size": [alto, ancho], "counts": counts.encode("ascii")}).astype(bool)

In [ ]:
%%writefile filamentos/metrica.py
"""Panoptic Quality (PQ) tal como la define la competición, más diagnósticos.

Un filamento predicho y uno real se emparejan si su IoU > 0.5. Con ese umbral
cada filamento solo puede emparejarse con uno del otro lado, así que no hace
falta un emparejamiento óptimo.

    PQ = (suma de IoU de los aciertos / TP) * (TP / (TP + FP/2 + FN/2))
"""
from dataclasses import dataclass, field

import numpy as np

UMBRAL_IOU = 0.5


@dataclass
class Resultado:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    suma_iou: float = 0.0
    # Diagnósticos que la rúbrica también mira
    uno_a_muchos: int = 0   # un filamento real cubierto por 2+ predichos (fragmentación)
    muchos_a_uno: int = 0   # un predicho que cubre 2+ reales (fusión)
    ious: list = field(default_factory=list)

    @property
    def sq(self):
        return self.suma_iou / self.tp if self.tp else 0.0

    @property
    def rq(self):
        den = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / den if den else 1.0

    @property
    def pq(self):
        return self.sq * self.rq

    def __iadd__(self, otro):
        for k in ("tp", "fp", "fn", "suma_iou", "uno_a_muchos", "muchos_a_uno"):
            setattr(self, k, getattr(self, k) + getattr(otro, k))
        self.ious += otro.ious
        return self


def _matriz_solape(reales, predichas):
    """Intersecciones y áreas usando etiquetas, sin comparar máscara contra máscara."""
    if not reales or not predichas:
        return np.zeros((len(reales), len(predichas))), None, None
    forma = reales[0].shape
    area_r = np.array([m.sum() for m in reales], float)
    area_p = np.array([m.sum() for m in predichas], float)
    inter = np.zeros((len(reales), len(predichas)))
    for i, r in enumerate(reales):
        # Recorte a la caja de r para no recorrer la imagen entera
        ys, xs = np.nonzero(r)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        rr = r[y0:y1, x0:x1]
        for j, p in enumerate(predichas):
            inter[i, j] = np.logical_and(rr, p[y0:y1, x0:x1]).sum()
    return inter, area_r, area_p


def evaluar_imagen(reales, predichas):
    """reales, predichas: listas de máscaras booleanas 2D del mismo tamaño."""
    res = Resultado()
    inter, area_r, area_p = _matriz_solape(reales, predichas)
    if area_r is None:
        res.fn, res.fp = len(reales), len(predichas)
        return res
    iou = inter / (area_r[:, None] + area_p[None, :] - inter)
    emparejados_r = set()
    emparejados_p = set()
    for i, j in zip(*np.nonzero(iou > UMBRAL_IOU)):
        emparejados_r.add(i)
        emparejados_p.add(j)
        res.suma_iou += iou[i, j]
        res.ious.append(float(iou[i, j]))
    res.tp = len(emparejados_r)
    res.fn = len(reales) - res.tp
    res.fp = len(predichas) - len(emparejados_p)

    # Un solape cuenta si cubre al menos el 10 % del filamento más pequeño de la pareja
    toca = inter > 0.1 * np.minimum(area_r[:, None], area_p[None, :])
    res.uno_a_muchos = int((toca.sum(axis=1) >= 2).sum())
    res.muchos_a_uno = int((toca.sum(axis=0) >= 2).sum())
    return res


def resumen(res):
    return (f"PQ {res.pq:.4f} (forma SQ {res.sq:.3f} x detección RQ {res.rq:.3f}) | "
            f"TP {res.tp} FP {res.fp} FN {res.fn} | "
            f"fragmentados {res.uno_a_muchos} fusionados {res.muchos_a_uno}")


def detalle_errores(reales, predichas):
    """Para cada filamento real y predicho: si acertó y su mejor IoU con el otro lado.

    Sirve para separar los errores "de verdad" (IoU casi 0: inventado o no visto) de los
    "casi aciertos" (IoU 0,1-0,5: lo vio, pero lo partió, lo fusionó o dibujó mal el borde).
    """
    inter, area_r, area_p = _matriz_solape(reales, predichas)
    if area_r is None:
        return [(False, 0.0)] * len(reales), [(False, 0.0)] * len(predichas)
    iou = inter / (area_r[:, None] + area_p[None, :] - inter)
    mejor_r, mejor_p = iou.max(axis=1), iou.max(axis=0)
    return ([(bool(v > UMBRAL_IOU), float(v)) for v in mejor_r],
            [(bool(v > UMBRAL_IOU), float(v)) for v in mejor_p])

In [ ]:
%%writefile filamentos/clasico.py
"""Línea base clásica: observador (zonas oscuras) + juez (reglas observadas).

Sin aprendizaje automático. Sirve para estrenar el formato de envío y como
referencia contra la que medir los modelos.
"""
from dataclasses import dataclass

import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage import measure, morphology


@dataclass
class Parametros:
    umbral_oscuro: float = 0.88    # intensidad relativa al fondo local
    radio_max: float = 0.90        # regla 2: nada más allá del 90 % del radio
    area_min: int = 330            # regla 3: tamaño mínimo
    oscuridad_min: float = 0.44    # regla 4: lo más negro son manchas solares
    alargamiento_min: float = 1.8  # regla 5: forma alargada
    union_px: int = 6              # trozos a menos de esta distancia se unen en un filamento


def disco_solar(img):
    """Centro y radio del disco a partir de los píxeles brillantes."""
    m = img > 0.3 * np.percentile(img, 99)
    m = ndi.binary_fill_holes(ndi.binary_opening(m, iterations=3))
    lab, _ = ndi.label(m)
    k = np.argmax(np.bincount(lab.ravel())[1:]) + 1
    yy, xx = np.nonzero(lab == k)
    return xx.mean(), yy.mean(), np.sqrt(len(xx) / np.pi)


def normalizar(img, cx, cy, R):
    """Divide por el fondo local (mediana en ventanas de ~100 px)."""
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    peque = np.asarray(Image.fromarray(relleno.astype(np.float32)).resize((512, 512), Image.BILINEAR))
    fondo = ndi.median_filter(peque, size=25)
    fondo = np.asarray(Image.fromarray(fondo).resize(img.shape[::-1], Image.BILINEAR))
    norm = img / np.maximum(fondo, 1)
    norm[~dentro] = 1
    return norm, r


def observador(norm, r, p):
    """Candidatos: zonas oscuras dentro del disco, uniendo trozos muy cercanos."""
    oscuro = (norm < p.umbral_oscuro) & (r < p.radio_max)
    oscuro = ndi.binary_opening(oscuro, iterations=1)
    # Los trozos cercanos se agrupan con una dilatación, pero la máscara final es la original
    grupos = measure.label(ndi.binary_dilation(oscuro, structure=morphology.disk(p.union_px)))
    return grupos * oscuro


def juez(etiquetas, norm, p):
    """Aplica las reglas a cada candidato y devuelve las máscaras aceptadas."""
    aceptadas = []
    for reg in measure.regionprops(etiquetas):
        if reg.area < p.area_min:
            continue
        y0, x0, y1, x1 = reg.bbox
        vals = norm[y0:y1, x0:x1][reg.image]
        if vals.min() < p.oscuridad_min:
            continue
        if reg.axis_major_length / max(reg.axis_minor_length, 1) < p.alargamiento_min:
            continue
        m = np.zeros(etiquetas.shape, bool)
        m[y0:y1, x0:x1] = reg.image
        aceptadas.append(m)
    return aceptadas


def predecir(ruta_imagen, p=Parametros()):
    img = np.asarray(Image.open(ruta_imagen).convert("L"), float)
    cx, cy, R = disco_solar(img)
    norm, r = normalizar(img, cx, cy, R)
    return juez(observador(norm, r, p), norm, p)

In [ ]:
%%writefile filamentos/observador.py
"""Observador con red neuronal: U-Net que predice, píxel a píxel, la probabilidad de filamento.

Entrada de 3 canales a resolución reducida:
  0. imagen original normalizada a [0, 1]
  1. imagen dividida por su fondo local (resalta lo oscuro respecto al entorno)
  2. distancia al centro del disco en radios (ayuda con la regla del borde)
Objetivo: media de las máscaras de todos los anotadores (0, 1/3, 1/2, 2/3 o 1).
"""
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

from . import clasico


def preparar_imagen(ruta, res):
    """Devuelve (entrada [3,res,res] float32, r [res,res])."""
    img = np.asarray(Image.open(ruta).convert("L").resize((res, res), Image.BILINEAR), np.float32)
    cx, cy, R = clasico.disco_solar(img)
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    # Fondo local: mediana calculada a 1/4 de resolución y reampliada (≈200 veces más rápido
    # que a resolución completa; el resultado cambia menos de un 1 % de media)
    k = 4
    peq = np.asarray(Image.fromarray(relleno).resize((res // k, res // k), Image.BILINEAR))
    fondo = ndi.median_filter(peq, size=max(3, res // 20 // k))
    fondo = np.asarray(Image.fromarray(fondo).resize((res, res), Image.BILINEAR))
    norm = np.clip(img / np.maximum(fondo, 1), 0, 1.5)
    norm[~dentro] = 1
    entrada = np.stack([img / 255.0, norm / 1.5, np.clip(r, 0, 1.2) / 1.2]).astype(np.float32)
    return entrada, r.astype(np.float32)


def objetivo_suave(anotadores, res, tam=2048):
    """Media de las máscaras de cada anotador, dibujadas directamente a resolución res."""
    escala = res / tam
    acum = np.zeros((res, res), np.float32)
    for anns in anotadores.values():
        lienzo = Image.new("L", (res, res), 0)
        d = ImageDraw.Draw(lienzo)
        for a in anns:
            for p in a["segmentation"]:
                if len(p) >= 6:
                    d.polygon([(p[i] * escala, p[i + 1] * escala) for i in range(0, len(p), 2)], fill=1)
        acum += np.asarray(lienzo, np.float32)
    return acum / max(len(anotadores), 1)


class DatosEntreno(torch.utils.data.Dataset):
    def __init__(self, entradas, objetivos, aumentar):
        self.x, self.y, self.aumentar = entradas, objetivos, aumentar

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x, y = self.x[i], self.y[i][None]
        if self.aumentar:
            k = np.random.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (1, 2))
            if np.random.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, :, ::-1]
            x = x.copy()
            x[0] = np.clip(x[0] * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))


def perdida(logits, y):
    """BCE + Dice suave: BCE aprende píxel a píxel, Dice compensa que el filamento ocupa poco."""
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    inter = (p * y).sum((2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((2, 3)) + y.sum((2, 3)) + 1)
    return bce + dice.mean()


@torch.no_grad()
def predecir_prob(modelo, entrada, dispositivo, tta=True):
    """Probabilidad [res,res]; con TTA promedia las 8 rotaciones/volteos."""
    x = torch.from_numpy(entrada)[None].to(dispositivo)
    vistas = [(k, f) for k in range(4) for f in (False, True)] if tta else [(0, False)]
    acum = 0
    for k, f in vistas:
        xi = torch.rot90(x, k, (2, 3))
        if f:
            xi = xi.flip(3)
        p = torch.sigmoid(modelo(xi))
        if f:
            p = p.flip(3)
        acum = acum + torch.rot90(p, -k, (2, 3))
    return (acum / len(vistas))[0, 0].float().cpu().numpy()


def a_instancias(prob, r, umbral=0.5, area_min=330, area_max=60000, radio_max=0.95, union_px=0, tam=2048):
    """Probabilidad a resolución reducida -> lista de máscaras booleanas a 2048x2048."""
    prob_g = np.asarray(Image.fromarray(prob).resize((tam, tam), Image.BILINEAR))
    r_g = np.asarray(Image.fromarray(r).resize((tam, tam), Image.BILINEAR))
    # Solo dentro del disco: fuera no hay filamentos y una red poco entrenada puede dar ruido
    fil = (prob_g > umbral) & (r_g < 1.0)
    if union_px:
        grupos, n = ndi.label(ndi.binary_dilation(fil, iterations=union_px))
        etiquetas = grupos * fil
    else:
        etiquetas, n = ndi.label(fil, structure=np.ones((3, 3)))
    mascaras = []
    for k, sl in enumerate(ndi.find_objects(etiquetas), start=1):
        if sl is None:
            continue
        m_local = etiquetas[sl] == k
        area = m_local.sum()
        # El filamento anotado más grande mide ~38 000 px; algo mucho mayor es un error
        if area < area_min or area > area_max:
            continue
        if r_g[sl][m_local].mean() > radio_max:
            continue
        m = np.zeros((tam, tam), bool)
        m[sl] = m_local
        mascaras.append(m)
    return mascaras

In [ ]:
%%writefile filamentos/detector.py
"""Detector + refinador: cada filamento como un objeto separado.

1. Detector (Mask R-CNN, preentrenado en COCO): a 1024 px propone cada filamento con
   una caja, una confianza y una silueta. Partir y unir lo decide él, no una regla.
2. Refinador (U-Net pequeña): recorta cada filamento detectado en la imagen original
   de 2048 px, con margen alrededor, y redibuja su borde a 256x256.

Arquitectura inspirada en el notebook público de Nomannic19 ("Mask R-CNN + U-Net
Refiner"); el código es nuestro.
"""
import numpy as np
import torch
from PIL import Image
from pycocotools import mask as mask_utils
from scipy import ndimage as ndi
from scipy.sparse.csgraph import connected_components

TAM = 2048
RECORTE = 256


# ---------------------------------------------------------------- detector

def construir_detector(preentrenado=True, res=1024):
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
    from torchvision.models.detection.anchor_utils import AnchorGenerator
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    from torchvision.ops import MultiScaleRoIAlign

    pesos = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if preentrenado else None
    m = maskrcnn_resnet50_fpn_v2(weights=pesos, weights_backbone=None)
    # Dos clases: fondo y filamento
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(
        m.roi_heads.mask_predictor.conv5_mask.in_channels, m.roi_heads.mask_predictor.conv5_mask.out_channels, 2)
    # Cajas desde 16 px y alargadas (los filamentos miden 2-6 veces más de largo que de ancho)
    m.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)),
                                             aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    # Siluetas a 28x28 en vez de 14x14: más detalle en filamentos finos
    m.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)
    m.roi_heads.detections_per_img = 100
    m.transform.min_size, m.transform.max_size = (res,), res
    return m


def cargar_gris(ruta, res):
    return np.asarray(Image.open(ruta).convert("L").resize((res, res), Image.BILINEAR), np.uint8)


def mascaras_res(anotaciones, res, tam=TAM):
    """Una máscara booleana [res,res] por filamento de un anotador.

    Se dibuja a 2048 con pycocotools (como la métrica oficial) y se reduce promediando.
    Dibujar el polígono directamente a baja resolución con PIL engorda los bordes: con
    máscaras perfectas, la PQ caía de 0,89 a 0,82 en una prueba con 15 imágenes."""
    out = []
    for a in anotaciones:
        rles = mask_utils.frPyObjects(a["segmentation"], tam, tam)
        m = mask_utils.decode(mask_utils.merge(rles)).astype(np.float32)
        m = np.asarray(Image.fromarray(m).resize((res, res), Image.BILINEAR)) > 0.5
        if m.any():
            out.append(m)
    return out


def _fotometria(img):
    """Cambios de brillo, contraste y gamma solo dentro del disco (fuera es negro)."""
    dentro = img > 0.01
    if np.random.rand() < 0.7:
        img[dentro] = img[dentro] ** np.random.uniform(0.8, 1.25)
    if np.random.rand() < 0.7:
        media = img[dentro].mean()
        img[dentro] = ((img[dentro] - media) * np.random.uniform(0.85, 1.15) + media) * np.random.uniform(0.9, 1.1)
    return np.clip(img, 0, 1)


class DatosDetector(torch.utils.data.Dataset):
    """Un ejemplo por (imagen, anotador): el detector aprende con siluetas nítidas de un experto,
    no con la media borrosa. Las versiones de una misma imagen comparten la imagen en memoria y
    las máscaras se dibujan al vuelo desde los polígonos (guardarlas todas no cabe en RAM)."""

    def __init__(self, ejemplos, res, aumentar):
        self.ejemplos, self.res, self.aumentar = ejemplos, res, aumentar  # [(imagen uint8 [res,res], anotaciones)]

    def __len__(self):
        return len(self.ejemplos)

    def __getitem__(self, i):
        img, anotaciones = self.ejemplos[i]
        return objetivo_aumentado(img, mascaras_res(anotaciones, self.res), self.aumentar)


def objetivo_aumentado(img, mascaras, aumentar):
    """img uint8 [H,W] + máscaras booleanas -> (tensor imagen, objetivo de Mask R-CNN), con giros y
    cambios de brillo si aumentar. Lo comparten el detector general y el especialista."""
    img = img.astype(np.float32) / 255
    m = np.stack(mascaras) if mascaras else np.zeros((0,) + img.shape, bool)
    if aumentar:
        k = np.random.randint(4)
        img, m = np.rot90(img, k), np.rot90(m, k, (1, 2))
        if np.random.rand() < 0.5:
            img, m = img[:, ::-1], m[:, :, ::-1]
        img = _fotometria(img.copy())
    img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
    cajas = np.zeros((len(m), 4), np.float32)
    for j, mj in enumerate(m):
        ys, xs = np.nonzero(mj)
        cajas[j] = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
    objetivo = {"boxes": torch.from_numpy(cajas), "labels": torch.ones(len(m), dtype=torch.int64),
                "masks": torch.from_numpy(m.astype(np.uint8))}
    return torch.from_numpy(img)[None].repeat(3, 1, 1), objetivo


def juntar(lote):
    return tuple(zip(*lote))


@torch.no_grad()
def detectar(modelo, img, disp, umbral_mascara=0.5):
    """img uint8 [res,res] -> (máscaras booleanas [N,res,res], confianzas [N]) sin filtrar por confianza."""
    modelo.eval()
    x = torch.from_numpy(img.astype(np.float32) / 255)[None].repeat(3, 1, 1).to(disp)
    s = modelo([x])[0]
    m = (s["masks"][:, 0] >= umbral_mascara).cpu().numpy()
    conf = s["scores"].float().cpu().numpy()
    activas = m.any(axis=(1, 2))
    return m[activas], conf[activas]


def a_rle(m):
    return mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))


def grupos_solapados(mascaras):
    """Índices agrupados: dos detecciones que se tocan (IoU > 0) van al mismo grupo."""
    n = len(mascaras)
    if n < 2:
        return [[i] for i in range(n)]
    rles = [a_rle(m) for m in mascaras]
    toca = np.asarray(mask_utils.iou(rles, rles, [0] * n)) > 0
    _, etiqueta = connected_components(toca, directed=False)
    return [list(np.flatnonzero(etiqueta == k)) for k in range(etiqueta.max() + 1)]


def a_2048(m):
    """Amplía con suavizado bilineal: el borde queda liso en vez de en escalones de 2 px."""
    return np.asarray(Image.fromarray(m.astype(np.float32)).resize((TAM, TAM), Image.BILINEAR)) > 0.5


def brillo_relativo(img, r):
    """Brillo de cada píxel dividido por el fondo local (mediana en ventanas de ~100 px a 2048).

    Mismo cálculo que el análisis de manchas (reglas 4 y 5): corrige el oscurecimiento hacia el
    borde y la iluminación desigual. 1 = como el fondo; 0,5 = la mitad de brillante.
    """
    img = img.astype(np.float32)
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    peque = np.asarray(Image.fromarray(relleno).resize((512, 512), Image.BILINEAR))
    fondo = ndi.median_filter(peque, size=25)
    fondo = np.asarray(Image.fromarray(fondo).resize(img.shape[::-1], Image.BILINEAR))
    rel = img / np.maximum(fondo, 1)
    rel[~dentro] = 1
    return rel.astype(np.float16)


def alargamiento(m):
    """Eje mayor / eje menor de la elipse equivalente (como skimage regionprops)."""
    ys, xs = np.nonzero(m)
    if len(xs) < 3:
        return 1.0
    l = np.linalg.eigvalsh(np.cov(np.stack([xs, ys]).astype(np.float64)))
    return float(np.sqrt(max(l[1], 1e-9) / max(l[0], 1e-9)))


def instancias(mascaras, conf, r, umbral=0.6, fusionar=True, area_min=0, radio_max=0.95,
               rel=None, oscuro_min=None, alarg_max=None):
    """Detecciones a [res,res] -> lista de máscaras a 2048 tras el juez.

    fusionar: las detecciones que se solapan se unen en un solo filamento (Mask R-CNN
    a veces propone el mismo filamento dos veces, o dos trozos de él).
    r: distancia al centro en radios a resolución res.
    oscuro_min (regla 4): se descarta si su punto más oscuro baja de este brillo relativo
        (rel = brillo_relativo); lo negrísimo suele ser una mancha solar.
    alarg_max (regla 5): se descarta si es más alargado que esto (hilos larguísimos).
    """
    keep = conf >= umbral
    mascaras = mascaras[keep]
    if fusionar:
        mascaras = [np.logical_or.reduce(mascaras[g]) for g in grupos_solapados(mascaras)]
    else:
        # Sin fusionar, un píxel solo puede pertenecer a un filamento: gana el de más confianza
        ocupado = np.zeros(r.shape, bool)
        limpias = []
        for m in mascaras:
            m = m & ~ocupado
            ocupado |= m
            if m.any():
                limpias.append(m)
        mascaras = limpias
    out = []
    escala2 = (TAM / r.shape[0]) ** 2
    for m in mascaras:
        if m.sum() * escala2 < area_min or r[m].mean() > radio_max:
            continue
        if oscuro_min is not None and float(rel[m].min()) < oscuro_min:
            continue
        if alarg_max is not None and alargamiento(m) > alarg_max:
            continue
        out.append(a_2048(m))
    return out


# ---------------------------------------------------------------- refinador

def caja_cuadrada(m, margen=1.8, minimo=96, tam=TAM):
    """Cuadrado centrado en el filamento, margen veces su lado mayor (contexto alrededor)."""
    ys, xs = np.nonzero(m)
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    lado = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * margen))
    lado = min(max(lado, minimo), tam)
    x0, y0 = int(round(cx - lado / 2)), int(round(cy - lado / 2))
    x0, y0 = min(max(x0, 0), tam - lado), min(max(y0, 0), tam - lado)
    return x0, y0, x0 + lado, y0 + lado


def recortar(img, caja, interp=Image.BILINEAR):
    x0, y0, x1, y1 = caja
    return np.asarray(Image.fromarray(img[y0:y1, x0:x1]).resize((RECORTE, RECORTE), interp))


def recortes_entreno(img2048, mascaras2048, aumentar):
    """Pares (recorte de imagen, recorte de máscara) a 256x256, uno por filamento.

    Con aumentar=True el margen y el centro varían un poco, para que el refinador no dependa
    de que el detector encuadre el filamento perfectamente."""
    out = []
    for m in mascaras2048:
        margen = np.random.uniform(1.5, 2.2) if aumentar else 1.8
        caja = caja_cuadrada(m, margen)
        if aumentar:
            lado = caja[2] - caja[0]
            dx, dy = np.random.randint(-lado // 10, lado // 10 + 1, 2)
            x0, y0 = min(max(caja[0] + dx, 0), TAM - lado), min(max(caja[1] + dy, 0), TAM - lado)
            caja = (x0, y0, x0 + lado, y0 + lado)
        out.append((recortar(img2048, caja), recortar(m.astype(np.uint8), caja, Image.NEAREST)))
    return out


class DatosRecortes(torch.utils.data.Dataset):
    def __init__(self, imagenes, mascaras):
        self.x, self.y = imagenes, mascaras  # uint8 [N,256,256]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x, y = self.x[i].astype(np.float32) / 255, self.y[i].astype(np.float32)
        k = np.random.randint(4)
        x, y = np.rot90(x, k), np.rot90(y, k)
        if np.random.rand() < 0.5:
            x, y = x[:, ::-1], y[:, ::-1]
        x = np.clip(x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.04, 0.04), 0, 1)
        return (torch.from_numpy(np.ascontiguousarray(x))[None].repeat(3, 1, 1),
                torch.from_numpy(np.ascontiguousarray(y))[None])


@torch.no_grad()
def refinar(refinador, img2048, mascaras2048, disp, umbral=0.85, lote=32, unir=True):
    """Redibuja cada filamento. Si el refinador no ve nada, se queda la silueta del detector.

    unir=False devuelve un filamento refinado por cada uno de entrada, aunque se pisen."""
    if not mascaras2048:
        return []
    refinador.eval()
    cajas = [caja_cuadrada(m) for m in mascaras2048]
    recs = np.stack([recortar(img2048, c) for c in cajas]).astype(np.float32) / 255
    probs = []
    for i in range(0, len(recs), lote):
        x = torch.from_numpy(recs[i:i + lote])[:, None].repeat(1, 3, 1, 1).to(disp)
        # Promedio de 4 vistas volteadas (TTA)
        p = 0
        for dims in ((), (3,), (2,), (2, 3)):
            xi = x.flip(dims) if dims else x
            pi = torch.sigmoid(refinador(xi).float())
            p = p + (pi.flip(dims) if dims else pi)
        probs.append((p / 4)[:, 0].cpu().numpy())
    probs = np.concatenate(probs)
    out = []
    for m, (x0, y0, x1, y1), p in zip(mascaras2048, cajas, probs):
        nueva = np.asarray(Image.fromarray(p).resize((x1 - x0, y1 - y0), Image.BILINEAR)) >= umbral
        if not nueva.any():
            out.append(m)
            continue
        # Nos quedamos con la parte conectada que toca la silueta del detector: el recorte puede
        # incluir trozos de filamentos vecinos que no son este
        etiquetas, _ = ndi.label(nueva)
        ids = np.unique(etiquetas[m[y0:y1, x0:x1] & (etiquetas > 0)])
        if not len(ids):
            out.append(m)
            continue
        nueva = np.isin(etiquetas, ids)
        completa = np.zeros((TAM, TAM), bool)
        completa[y0:y1, x0:x1] = nueva
        out.append(completa)
    if not unir:
        return out
    # Si dos filamentos refinados se pisan, se unen (igual que en el detector)
    return [np.logical_or.reduce([out[i] for i in g]) for g in grupos_solapados(out)]

In [ ]:
%%writefile filamentos/candidatos.py
"""Paso 1 del tribunal: el detector en modo generoso y la ficha de cada candidato.

El detector de la ronda 3 propone todo lo que ve con confianza >= conf_min (en vez de 0,8). De cada
candidato se guarda su silueta (la del detector y la del refinador, cada una por separado, sin unir
solapes), una ficha de rasgos y, si se pide, un recorte para el juez A. Con eso se ajustan las perillas
del tribunal fuera de Kaggle, sin GPU (docs/arquitectura-tribunal.md).
"""
import numpy as np
from PIL import Image
from pycocotools import mask as mask_utils
from scipy import ndimage as ndi

from . import clasico
from . import detector as D

TAM = D.TAM
FRANJAS = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.01)


def radio(img):
    """Distancia al centro del disco solar, en radios, para cada píxel."""
    cx, cy, R = clasico.disco_solar(img.astype(np.float32))
    yy, xx = np.indices(img.shape)
    return (np.hypot(xx - cx, yy - cy) / R).astype(np.float32)


def a_texto(m):
    r = D.a_rle(m)
    return r["counts"].decode("ascii")


def de_texto(counts):
    return {"size": [TAM, TAM], "counts": counts.encode("ascii")}


def _ficha(m, rel, r2048, conf, m_res):
    ys, xs = np.nonzero(m)
    y0, y1, x0, x1 = max(ys.min() - 12, 0), ys.max() + 13, max(xs.min() - 12, 0), xs.max() + 13
    trozo = m[y0:y1, x0:x1]
    anillo = ndi.binary_dilation(trozo, iterations=10) & ~ndi.binary_dilation(trozo, iterations=2)
    dentro = rel[y0:y1, x0:x1][trozo]
    fuera = rel[y0:y1, x0:x1][anillo]
    med_fuera = float(np.median(fuera)) if fuera.size else 1.0
    return dict(conf=round(float(conf), 4), area=int(m.sum()), cx=round(float(xs.mean()), 1),
                cy=round(float(ys.mean()), 1), ancho=int(x1 - x0), alto=int(y1 - y0),
                radio=round(float(r2048[m].mean()), 4), osc_med=round(float(np.median(dentro)), 4),
                osc_p5=round(float(np.percentile(dentro, 5)), 4), anillo=round(med_fuera, 4),
                contraste_local=round(float(np.median(dentro)) / max(med_fuera, 1e-3), 4),
                alarg=round(D.alargamiento(m_res), 3))


def procesar(gen, ref, grande, disp, res=1024, conf_min=0.1, umbral_ref=0.5, lado_recorte=0):
    """Una imagen a 2048 -> (candidatos, recortes, predicción de la ronda 3 para comprobar).

    candidatos: lista de dict con la ficha, 'rle' (detector) y 'rle_ref' (refinador).
    recortes: uint8 [N, 2, lado, lado] (imagen y silueta) o None si lado_recorte=0.
    """
    import torch

    img = np.asarray(Image.fromarray(grande).resize((res, res), Image.BILINEAR), np.uint8)
    r = radio(img)
    r2048 = np.asarray(Image.fromarray(r).resize((TAM, TAM), Image.BILINEAR))
    rel = D.brillo_relativo(grande, r2048).astype(np.float32)
    with torch.autocast(disp, enabled=disp == "cuda"):
        m, c = D.detectar(gen, img, disp)
    # Lo mismo que la ronda 3 (juez 0,8 uniendo solapes + refinador 0,5), para comprobar que todo cuadra
    r3 = D.instancias(m, c, r, umbral=0.8, fusionar=True, area_min=0)
    with torch.autocast(disp, enabled=disp == "cuda"):
        r3 = D.refinar(ref, grande, r3, disp, umbral=umbral_ref)
    keep = c >= conf_min
    m, c = m[keep], c[keep]
    m2048 = [D.a_2048(x) for x in m]
    keep = [i for i, x in enumerate(m2048) if x.any()]
    m, c, m2048 = m[keep], c[keep], [m2048[i] for i in keep]
    with torch.autocast(disp, enabled=disp == "cuda"):
        refinadas = D.refinar(ref, grande, m2048, disp, umbral=umbral_ref, unir=False)
    cands, recs = [], []
    for mi, ci, mg, mr in zip(m, c, m2048, refinadas):
        f = _ficha(mg, rel, r2048, ci, mi)
        f["area_ref"] = int(mr.sum())
        f["rle"], f["rle_ref"] = a_texto(mg), a_texto(mr)
        cands.append(f)
        if lado_recorte:
            caja = D.caja_cuadrada(mg)
            x0, y0, x1, y1 = caja
            a = np.asarray(Image.fromarray(grande[y0:y1, x0:x1]).resize((lado_recorte,) * 2, Image.BILINEAR))
            b = np.asarray(Image.fromarray(mg[y0:y1, x0:x1].astype(np.uint8) * 255).resize(
                (lado_recorte,) * 2, Image.NEAREST))
            recs.append(np.stack([a, b]))
    recortes = np.stack(recs).astype(np.uint8) if recs else (
        np.zeros((0, 2, lado_recorte, lado_recorte), np.uint8) if lado_recorte else None)
    return cands, recortes, r3


def duplicados(cands, clave="rle_ref", umbral=0.5):
    """True para los candidatos que repiten (IoU > umbral) a otro de más confianza."""
    if not cands:
        return np.zeros(0, bool)
    orden = np.argsort([-x["conf"] for x in cands], kind="stable")
    rles = [de_texto(cands[i][clave]) for i in orden]
    iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)))
    dup_ord = np.triu(iou > umbral, 1).any(0)
    dup = np.zeros(len(cands), bool)
    dup[orden] = dup_ord
    return dup


def tabla_franjas(cands_por_imagen, reales_por_imagen, clave="rle_ref"):
    """Acierto por franja de confianza y cuántos filamentos reales quedan al alcance.

    cands_por_imagen: {imagen: [candidato]}; reales_por_imagen: {imagen: [[RLE de un experto], ...]}.
    Cada candidato se compara con cada experto por separado, como la métrica. Los candidatos que
    repiten a otro de más confianza (IoU > 0,5) no cuentan: el tribunal se quedaría con uno.
    Devuelve (filas por franja, reales totales, reales que algún candidato alcanza con IoU > 0,5,
    duplicados descartados).
    """
    franjas = np.asarray(FRANJAS)
    aciertos = np.zeros(len(franjas) - 1)
    total = np.zeros(len(franjas) - 1)
    suma_iou = np.zeros(len(franjas) - 1)
    n_reales = alcanzables = n_dup = 0
    for n, cands in cands_por_imagen.items():
        dup = duplicados(cands, clave)
        n_dup += int(dup.sum())
        cands = [x for x, d in zip(cands, dup) if not d]
        rles = [de_texto(x[clave]) for x in cands]
        conf = np.array([x["conf"] for x in cands])
        k = np.clip(np.searchsorted(franjas, conf, side="right") - 1, 0, len(franjas) - 2)
        for reales in reales_por_imagen[n]:
            n_reales += len(reales)
            if not rles:
                continue
            if reales:
                iou = np.asarray(mask_utils.iou(rles, reales, [0] * len(reales)))
                mejor = iou.max(1)
                alcanzables += int((iou.max(0) > 0.5).sum())
            else:
                mejor = np.zeros(len(rles))
            np.add.at(total, k, 1)
            np.add.at(aciertos, k, mejor > 0.5)
            np.add.at(suma_iou, k, np.where(mejor > 0.5, mejor, 0))
    filas = []
    for i in range(len(franjas) - 1):
        if total[i]:
            filas.append(dict(franja=f"{franjas[i]:.2f}-{min(franjas[i + 1], 1):.2f}", candidatos=int(total[i]),
                              aciertos=int(aciertos[i]), pct_acierto=round(float(100 * aciertos[i] / total[i]), 1),
                              iou_medio=round(float(suma_iou[i] / max(aciertos[i], 1)), 3)))
    return filas, n_reales, alcanzables, n_dup

## Datos, separación estudio / examen y pesos de la ronda 3
Misma separación que las rondas 1-4.

In [ ]:
import glob, hashlib, json, time
import numpy as np, pandas as pd, torch
from PIL import Image
from filamentos.datos import cargar_anotaciones, mascaras_de
from filamentos.metrica import Resultado, evaluar_imagen, resumen
from filamentos import detector as D, candidatos as C

disp = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()

def buscar(patron):
    for raiz in ('/kaggle/input', os.environ.get('MAGFILO', '.'), os.environ.get('PESOS', '.')):
        r = glob.glob(f'{raiz}/**/{patron}', recursive=True)
        if r:
            return r[0]
    return None

json_path = buscar('MAGFiLO_1.0_Annotations_kaggle2026_train.json')
assert json_path, 'No encuentro los datos: añade la competición como Input del notebook'
DATOS = os.path.dirname(os.path.dirname(json_path))
PESOS_DET, PESOS_REF = buscar('detector_mejor.pt'), buscar('refinador.pt')
if not PRUEBA_RAPIDA:
    assert PESOS_DET and PESOS_REF, ('No encuentro los pesos de la ronda 3: añade como Input el Output del '
                                     'notebook filamentos-detector-ronda-2. Hay en /kaggle/input: '
                                     + str(glob.glob('/kaggle/input/*/*')))
print('Datos en', DATOS, '| pesos:', PESOS_DET, PESOS_REF)

ANOT = cargar_anotaciones(DATOS)
nombres = sorted(n for n in ANOT if os.path.exists(f'{DATOS}/train/train_images/{n}'))

def es_val(n):
    return int(hashlib.md5(n.encode()).hexdigest(), 16) % 100 < 15
val = [n for n in nombres if es_val(n)]
tr = [n for n in nombres if not es_val(n)]
if PRUEBA_RAPIDA:
    tr, val = tr[:3], val[:3]
print(len(tr), 'imágenes de estudio,', len(val), 'de examen')

gen = D.construir_detector(False, RES).to(disp)
ref = smp.Unet('resnet18', encoder_weights=None, in_channels=3, classes=1).to(disp)
if PESOS_DET:
    gen.load_state_dict(torch.load(PESOS_DET, map_location=disp))
    ref.load_state_dict(torch.load(PESOS_REF, map_location=disp))

## Candidatos del examen (con recortes) y comprobación de la ronda 3

In [ ]:
def reales_rle(n):
    return [[D.a_rle(m) for m in mascaras_de(anns)] for anns in ANOT[n].values()]

def gris2048(n):
    return np.asarray(Image.open(f'{DATOS}/train/train_images/{n}').convert('L'), np.uint8)

cand_val, recortes, indice, reales_val = {}, [], [], {}
r3 = Resultado()
t = time.time()
for i, n in enumerate(val):
    grande = gris2048(n)
    cands, recs, pred_r3 = C.procesar(gen, ref, grande, disp, RES, CONF_MIN, UMBRAL_REF, LADO_RECORTE)
    cand_val[n] = cands
    recortes.append(recs)
    indice += [(n, k) for k in range(len(cands))]
    reales_val[n] = reales_rle(n)
    for rles in reales_val[n]:
        r3 += evaluar_imagen([D.mask_utils.decode(r).astype(bool) for r in rles], pred_r3)
    if i % 20 == 0:
        print(f'{i + 1}/{len(val)} imágenes, {sum(map(len, cand_val.values()))} candidatos, {time.time() - t:.0f} s', flush=True)
print('Cadena de la ronda 3 (debe dar ~0,4405):', resumen(r3))
print(f'{sum(map(len, cand_val.values()))} candidatos en el examen ({time.time() - t:.0f} s)')

np.savez_compressed('recortes_examen.npz', recortes=np.concatenate(recortes),
                    imagen=np.array([a for a, _ in indice]), k=np.array([b for _, b in indice]))
with open('candidatos_examen.json', 'w') as fh:
    json.dump(cand_val, fh, separators=(',', ':'))

## La tabla: acierto por franja de confianza
Cada candidato se compara con cada experto, como la métrica. **% acierto** se compara con la raya del tribunal:
PQ ÷ (2 × IoU medio) ≈ 32 %. **Alcanzables** = filamentos reales que algún candidato dibuja con IoU > 0,5: es el
techo de lo que el tribunal puede rescatar.

In [ ]:
pd.set_option('display.width', 200)
for clave, nombre in (('rle_ref', 'con refinador'), ('rle', 'silueta del detector')):
    filas, n_reales, alcanzables, n_dup = C.tabla_franjas(cand_val, reales_val, clave)
    print(f'\n=== Examen, {nombre} ({n_dup} candidatos repetidos descartados) ===')
    print(pd.DataFrame(filas).to_string(index=False))
    print(f'Filamentos reales: {n_reales} | al alcance de algún candidato: {alcanzables} '
          f'({100 * alcanzables / n_reales:.0f} %) | la ronda 3 acertaba {r3.tp}')

## Candidatos de las imágenes de estudio
Sin recortes. El detector las estudió, así que aquí lo verá todo más fácil: sirven para comparar, no para medir.

In [ ]:
if CON_ESTUDIO:
    cand_tr, t = {}, time.time()
    for i, n in enumerate(tr):
        cand_tr[n], _, _ = C.procesar(gen, ref, gris2048(n), disp, RES, CONF_MIN, UMBRAL_REF, 0)
        if i % 100 == 0:
            print(f'{i + 1}/{len(tr)} imágenes, {time.time() - t:.0f} s', flush=True)
    with open('candidatos_estudio.json', 'w') as fh:
        json.dump(cand_tr, fh, separators=(',', ':'))
    filas, n_reales, alcanzables, n_dup = C.tabla_franjas(cand_tr, {n: reales_rle(n) for n in tr})
    print('\n=== Estudio, con refinador (el detector ya las conoce) ===')
    print(pd.DataFrame(filas).to_string(index=False))
    print(f'Filamentos reales: {n_reales} | al alcance: {alcanzables} ({100 * alcanzables / n_reales:.0f} %)')
print(f'Tiempo total: {(time.time() - T0) / 60:.0f} min')